# Optimizing Pointing

<!-- TODO: Link to a streamlined version in notebook and/or script -->
In this notebook, we will explore the `jwpoint` functionality that can
help us find the optimal pointing when planning an upcoming JWST observations.

The goal here, for a given instrument, detector and subaperture,
is to select the optimal pointing position that minimizes the amount of bad pixels
and respects a series of other criteria.

<!-- TODO: Link to pointing API docs -->
This is mostly done via `pointing.do_region_search()`.
We will explain its arguments in the following sections.

## Downloading a reference observation

<!-- TODO: Have reference observation library (could be just download links or file IDs) -->
Before we start optimizing the pointing, we need to download a reference observation.
We need an observation with the same instrument, detector, subaperture and filter as our planned observation.
Ideally, there should be at least once easy to find and isolated point source in the observation so we can use it as a reference PSF.

Here we use an data from the 1205 program which observed high-z quasars with NIRCam
We use an observation from the long-wavelength (LW) channel in the F430M filter and the SUB400P subarray.

In [ ]:
from pathlib import Path
from astroquery.mast import Observations

data_dir = Path("data")
program_dir = "01205"

filename = Path("jw01205002001_02104_00002_nrcblong_cal.fits")
uri = f"mast:JWST/product/{filename}"

filepath = data_dir / program_dir / filename
_ = Observations.download_file(uri, local_path=filepath)

In [ ]:
import numpy as np
from astropy.io import fits

with fits.open(filepath) as hdul:
    hdr = hdul[0].header
    img = hdul[1].data
dq_mask = np.isnan(img)

In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt

rcParams["image.origin"] = "lower"

plt.imshow(img, norm="symlog")
plt.xlabel("X [pixel]")
plt.ylabel("Y [pixel]")
plt.show()

It is pretty clear which is our point source target here. Let us extract it so it can serve as a reference PSF.

## Extracting the reference PSF

<!-- TODO: Link to API -->
First, we can use `pointing.apply_pointing()` to get the position of our target.

In [ ]:
from jwpoint import pointing
x_target, y_target = (int(round(off)) for off in pointing.apply_pointing(hdr["XOFFSET"], hdr["YOFFSET"], filepath))

In [ ]:
from matplotlib import rcParams
import matplotlib.pyplot as plt

rcParams["image.origin"] = "lower"

plt.imshow(img, norm="symlog")
plt.plot(x_target, y_target, "r*")
plt.xlabel("X [pixel]")
plt.ylabel("Y [pixel]")
plt.show()

In [ ]:
from jwpoint.plot import plot_dithers, zoom_plot

zoom_plot(img, x_target, y_target, size=64, show_mask=False)
plt.xlabel("X [pixel]")
plt.ylabel("Y [pixel]")
plt.show()

This looks good! We can crop the image.
The PSF must have the same shape as the region size we will be optimizing later.
Here we set this to 70 pixels, which is a reasonable value if what we care about is the close surroundings of the source.

In [ ]:
crop_size = 70
hs = crop_size // 2
psf_with_badpix = img[y_target-hs:y_target+hs, x_target-hs:x_target+hs]

In [ ]:
plt.imshow(psf_with_badpix, norm="symlog")
plt.xlabel("X [pixel]")
plt.ylabel("Y [pixel]")
plt.show()

This cropped PSF has bad pixels set to NaN.
To avoid propagating NaNs throughout our calculations, we can replace them with a median filter.

In [ ]:
psf = pointing.filter_crop(psf_with_badpix)

In [ ]:
plt.imshow(psf, norm="symlog")
plt.xlabel("X [pixel]")
plt.ylabel("Y [pixel]")
plt.show()

## Optimal region search

The `pointing.do_region_search()` function uses the DQ (bad pixel) mask from the reference image to find regions in the data
with as little bad pixels as possible.
It does a naive grid search over all possible regions in the image.

By default, for each position, it defines a region of size `crop_size` and calculates the number of bad pixels.
The `n_top` best regions are then returned.
Let us test this for a single region.

In [ ]:
x_naive, y_naive = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=1
)

Let us break down what is shown here:

On the left of the first plot, we se the bad pixel mask for the entire subaperture.
In the middle, we see the bad pixel which is color-coded by the number of bad-pixels for a 70x70 region centered on this pixels.
In short, the darker a point is, the better.
On the right, we see the science image for reference.
On each panel the "1" indicates the best pointing position

In the second plot, we see a zoom on this optimal region.
The top panel shows the actual region in the image
while the bottom shows the reference PSF with the bad pixel mask from this region.
The latter is useful to get a sense of where bad pixels would fall on a point source.

## Searching for multiple regions

If we wanted to get a few options, we could simply increase `n_top`.

In [ ]:
x_few, y_few = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=5
)
x_few, y_few

The x and Y positions are each returned in a NumPy array.

## Weighting bad pixels with a PSF

The searches above simply summed the number of bad pixels everywhere in the image uniformly.
While this can work, it may be desirable to penalize bad pixels near the core of the PSF more than on the edges.
This can be done with the `kernel="weighted"` argument:

In [ ]:
x_weighted, y_weighted = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=1, kernel="weighted"
)

As we can see, in this case, it barely changes the result, but it does change how pixels are weighted (we see PPSFs instead of squares).

## Rejecting bad pixels at the core of the PSF

Sometimes, the weighting from the previous section may not be enough.
There are cases where any region with bad pixels in the core of the PSF should be rejected.
This can be done with the `forbidden_size` argument, which defines the size of the PSF core
where no bad pixels are accepted.

In [ ]:
x_forbid, y_forbid = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=1, forbidden_size=25,
)

As we can see, this changes the results quite a bit: many pixels are now rejected (NaN in the middle panel)
and the pointing goes from middle left to bottom right.

However, be careful not to push this option too far: you will reach a regime where no region satisfies the requirement.

In [ ]:
# TODO: Update error to be more useful or print that none found
from importlib import reload
reload(pointing)
x_forbid, y_forbid = pointing.do_region_search(
    dq_mask,
    img,
    crop_size,
    psf=psf,
    n_top=1,
    forbidden_size=40,
)

## Requiring a minimal edge distance

If we need our target to be centered in the subarray,
or if we don't want to take too many risks by putting it near the edge,
we can request a minimum edge distance for the optimal region(s).
By default, the search window (region size of `crop_size = 70` here) is used as the minimum edge distance.

In [ ]:
x_edge, y_edge = pointing.do_region_search(
    dq_mask,
    img,
    crop_size,
    psf=psf,
    n_top=1,
    min_edge_distance=150,
)

## Accounting for the subarray in the short-wavelength channel

In all the searches above, we considered only the long-wavelength (LW) channel with the F430M filter.
However, NIRCam also observed in the short-wavelength (SW) channel simultaneously.

### Naive LW transformed to SW

The simplest thing we can do is just take our optimal pointing for the LW channel and see where it falls in SW.
Let us first extract the LW filter and subarray for later use.

In [ ]:
filt = hdr["FILTER"]
subarray = hdr["SUBARRAY"]
print(f"Filter: {filt}")
print(f"Subarray: {subarray}")

And infer the SW detector from the initial "naive" optimal position.
<!-- TODO: Link to docs or "understanding" notebook -->

In [ ]:
detector_sw = pointing.get_sw_detector(x_naive, y_naive, subarray)
print(f"Short wavelength detector: {detector_sw}")

We can now download the SW file and calculate where the naive pointing falls on it.
<!-- TODO: Link to docs or "understanding" notebook -->

In [ ]:
filename_sw = str(filename).replace("nrcblong", detector_sw)

uri = f"mast:JWST/product/{filename_sw}"

filepath_sw = data_dir / program_dir / filename_sw
_ = Observations.download_file(uri, local_path=filepath_sw)

In [ ]:
with fits.open(filepath_sw) as hdul_sw:
    hdr_sw = hdul_sw[0].header
    img_sw = hdul_sw[1].data

filt_sw = hdr_sw["FILTER"]

In [ ]:
x_naive_sw, y_naive_sw = pointing.long_to_short(x_naive, y_naive, filepath, filepath_sw)

In [ ]:
fig, axs = plt.subplots(1, 2, figsize=(10, 5))
ax_lw, ax_sw = axs
ax_lw.imshow(img, norm="symlog")
ax_lw.plot(x_naive, y_naive, "r*", label="naive position")
ax_lw.set_xlabel("X [pixel]")
ax_lw.set_ylabel("Y [pixel]")
ax_lw.set_title(f"LW image in filter {filt}")

ax_sw.imshow(img_sw, norm="symlog")
ax_sw.plot(x_naive_sw, y_naive_sw, "r*")
ax_sw.set_xlabel("X [pixel]")
ax_sw.set_ylabel("Y [pixel]")
ax_sw.set_title(f"SW Image in filter {filt_sw}")
fig.legend()
plt.show()

Oups! As we can see above, the pointing falls outside the subarray in the SW filter.
This can happen because some subarrays (such as `SUB400P` used here) have different FOVs in the LW and SW channels.
If we don't care about the SW data, this is not a big deal, but if we do we must make sure our optimal position
is usable in both channels.

### Using the subarray option

Currently, `jwpoint` does not support joint optimization in the LW and SW filters since most of the science cases we used it for
had a strong emphasis on LW.
However, it should not be too hard to add, so feel free to open [an issue](https://github.com/vandalt/jwpoint/issues/new/choose) or a [pull request](https://github.com/vandalt/jwpoint/compare) on GitHub!

For now, what we do have is a `subarray` argument that will restrict the search to regions that are visible in both channels for a given subarray,
but the optimization takes place entirely in the provided LW image.

In [ ]:
x_naive_sub, y_naive_sub = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=1, subarray=subarray,
)

Again, we can propagate this to the SW filter and see where it falls on both detectors.

In [ ]:
x_naive_sub_sw, y_naive_sub_sw = pointing.long_to_short(x_naive_sub, y_naive_sub, filepath, filepath_sw)

fig, axs = plt.subplots(1, 2, figsize=(10, 5))
ax_lw, ax_sw = axs
ax_lw.imshow(img, norm="symlog")
ax_lw.plot(x_naive_sub, y_naive_sub, "r*", label="Subarray-constrained position")
ax_lw.set_xlabel("X [pixel]")
ax_lw.set_ylabel("Y [pixel]")
ax_lw.set_title(f"LW image in filter {filt}")

ax_sw.imshow(img_sw, norm="symlog")
ax_sw.plot(x_naive_sub_sw, y_naive_sub_sw, "r*")
ax_sw.set_xlabel("X [pixel]")
ax_sw.set_ylabel("Y [pixel]")
ax_sw.set_title(f"SW Image in filter {filt_sw}")

fig.legend()
plt.show()

Looks like the contraint worked and the optimal position found is now within both detectors.

## Optimizing pointing for multiple dithers

In Understanding Pointing, we discuss how to calculate the pointing for multiple dithers.
Similarly, we can optimize the pointing by jointly optimizing the number of bad pixels
across dithers instead of on a region.

This is done by specifying the `joint_offsets` argument.
It needs to be a list of tuples, each with the `x` and `y` coordinates of the dithers.

Let us say we wanted to apply the [INTRAMODULEBOX](https://jwst-docs.stsci.edu/jwst-near-infrared-camera/nircam-operations/nircam-dithers-and-mosaics/nircam-primary-dithers#NIRCamPrimaryDithers-INTRAMODULEBOXdithers&gsc.tab=0) pattern.
We first need to fetch the dither offsets and convert them to pixels instead of arcsec.

**Note: We are working on making the multi-dither functionality more polished and easier to use**
<!-- TODO: Pass only pattern and number of dithers to function -->
<!-- TODO: Plots with dither automatically in search -->


In [ ]:
from jwpoint.dithers import get_dither_info

# TODO: Refactor so can pass pattern name and number of points?
# TODO: Reconcile format of dither offsets and what is returned by this so can pass more easily?
# TODO: Get the offsets in pixels?
dither_pattern = "INTRAMODULEBOX"

pscale = pointing.PSCALE_DICT[hdr["DETECTOR"]]

dither_df = get_dither_info(dither_pattern, ndithers=5)
dither_df["x_pix"] = dither_df.x / pscale
dither_df["y_pix"] = dither_df.y / pscale
dither_offsets = [xy for xy in zip(dither_df.x_pix, dither_df.y_pix)]

Then we can pass the offsets so that they are taken into account for the region search.
Note that the plots currently only show the reference pointing position and do not show the dithers,
the dithers are however considered when counting bad pixels.

In [ ]:
x_dithers, y_dithers = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=1, joint_offsets=dither_offsets
)

We can plot the dithers ourselves in a subsequent test

In [ ]:
x_dithers_all = x_dithers + dither_df.x_pix.astype(int)
y_dithers_all = y_dithers + dither_df.y_pix.astype(int)
plot_dithers(img, xopt_all=x_dithers_all, yopt_all=y_dithers_all, size=crop_size, psf=psf)
plt.show()

And we can also do this but optimize for the `n_top` best regions while accounting for dithers

In [ ]:
x_dithers_multi, y_dithers_multi = pointing.do_region_search(
    dq_mask, img, crop_size, psf=psf, n_top=2, joint_offsets=dither_offsets
)

Again, we need to plot the dithers manually for now.

In [ ]:
for i, (x, y) in enumerate(zip(x_dithers_multi, y_dithers_multi)):
    print(f"Optimal pointing {i+1}")
    x_all = x + dither_df.x_pix.astype(int)
    y_all = y + dither_df.y_pix.astype(int)
    plot_dithers(img, xopt_all=x_all, yopt_all=y_all, size=crop_size, psf=psf)
    plt.show()